In [ ]:
# 실습 준비 — 13주차 데이터통계분석 전체 돌아보기
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch4_scores400.csv", "ch11_potato.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 연속형 확률변수

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%precision 3
%matplotlib inline

In [ ]:
from scipy import integrate
import warnings

# 적분에 관한 warning을 출력하지 않도록 한다
warnings.filterwarnings('ignore', category=integrate.IntegrationWarning)

불공정한 룰렛의 확률 밀도함수
$$
f(x) = \left\{
\begin{array}{ll}
2x && (0 \le x \le 1) \\
0 && (otherwise)
\end{array}
\right.
$$


$$
P(0.4 \le X \le 0.6) = \int_{0.4}^{0.6} 2x \, dx
$$

In [ ]:
def f(x):
    if 0 <= x <= 1:
        return 2 * x
    else:
        return 0

In [ ]:
xs = np.linspace(0, 1, 100)
xy = [f(x) for x in xs]
plt.plot(xs, xy)
plt.fill_between(xs, xy, where=(xs >= 0.4) & (xs <= 0.6))
plt.show()

In [ ]:
integrate.quad(f, 0.4, 0.6) # 첫번째: 적분값, 두번째: 추정오차

In [ ]:
integrate.quad(f, -np.inf, np.inf)[0]

#### 기댓값

$$
\mu = E(X) = \int_{-\infty}^{\infty} xf(x)dx
$$

In [ ]:
mean = integrate.quad(lambda x: x*f(x), -np.inf, np.inf)[0]
mean

#### 분산
$$
\sigma^2 = V(X) = \int_{-\infty}^{\infty}(x-\mu)^2f(x)dx
$$

In [ ]:
integrate.quad(lambda x: (x-mean)**2*f(x), -np.inf, np.inf)[0]

# 대표적인 연속형 확률분포

In [ ]:
from scipy import stats

#### 정규분포
$
N(2, 0.5^2)
$

In [ ]:
# 정규분포
rv = stats.norm(2, 0.5)

In [ ]:
# 기댓값, 분산
rv.mean(), rv.var()

#### 표준정규분포
N(0, 1)

In [ ]:
rv = stats.norm()

X가 -1보다 작을 확률

$P(X\le -1)$

In [ ]:
rv.cdf(-1)

In [ ]:
xs = np.linspace(-3, 3, 100)
xy = rv.pdf(xs)
plt.plot(xs, xy)
plt.fill_between(xs, xy, where=(xs <= -1))
plt.show()

상위 $\alpha$인 지점(좌표)

$P(Z\ge z_\alpha) = \alpha$

In [ ]:
# 상위 30%인 지점
rv.isf(0.3)

In [ ]:
xs = np.linspace(-3, 3, 100)
xy = rv.pdf(xs)
plt.plot(xs, xy)
plt.fill_between(xs, xy, where=(xs >= rv.isf(0.3)))
plt.plot(rv.isf(0.3), 0, 'ro')
plt.show()

In [ ]:
# 90% 구간
rv.interval(0.9)

In [ ]:
xs = np.linspace(-3, 3, 100)
xy = rv.pdf(xs)
plt.plot(xs, xy)
plt.fill_between(xs, xy, where=((xs >= rv.interval(0.9)[0]) & (xs <= rv.interval(0.9)[1])))
plt.show()

#### t분포
$t(n)$

In [ ]:
n = 10
rv = stats.t(n)

In [ ]:
xs = np.linspace(-3, 3, 100)
xy = rv.pdf(xs)
plt.plot(xs, xy)

# 통계정 추정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
work_dir = "/content/drive/MyDrive/데이터통계분석/source/data"
# work_dir = "{나의 data 경로}"
os.chdir(work_dir)

In [ ]:
import pandas as pd
df = pd.read_csv('ch4_scores400.csv')
df.head()

In [ ]:
n = 20
np.random.seed(0)
sample = np.random.choice(df['score'], n)
s_mean = np.mean(sample)
n, s_mean

#### 모분산을 알고 있는 경우

In [ ]:
p_var = df['score'].var(ddof=0) # 모분산
rv = stats.norm()
lcl = s_mean - rv.isf(0.025) * np.sqrt(p_var/n)
ucl = s_mean - rv.isf(0.975) * np.sqrt(p_var/n)
lcl, ucl

#### 모분산을 모르는 경우


In [ ]:
s_var = np.var(sample, ddof=1) # 불편분산
rv = stats.t(n-1)
lcl = s_mean - rv.isf(0.025) * np.sqrt(s_var/n)
ucl = s_mean - rv.isf(0.975) * np.sqrt(s_var/n)
lcl, ucl

# 통계적 가설검정

In [ ]:
df = pd.read_csv('ch11_potato.csv')
df.head()

#### 모분산을 알고 있는 경우
귀무가설: 모평균이 130g 이다.\
대립가설: 모평균이 130g 이 아니다.\
모분산 = 9
표본갯수 = 14

In [ ]:
n = len(df)
sample = df['무게']
s_mean = sample.mean()
z = (s_mean - 130) / np.sqrt(9/n)
z

In [ ]:
# 단측검정
rv = stats.norm()
rv.isf(0.95)

In [ ]:
# 단측검정
rv.cdf(z)

In [ ]:
# 양측검정
rv.interval(0.95)

In [ ]:
# 양측검정
rv.cdf(z) * 2

#### 모분산을 모르고 있는 경우

In [ ]:
s_var = np.var(sample, ddof=1)
t = (s_mean - 130) / np.sqrt(s_var/n)
t

In [ ]:
# 단측검정
rv = stats.t(n-1)
rv.isf(0.95)

In [ ]:
# 단측검정
rv.cdf(t)

In [ ]:
# 양측검정
rv.interval(0.95)

In [ ]:
# 양측검정
rv.cdf(t) * 2

In [ ]:
t, p = stats.ttest_1samp(sample, 130)
t, p